# fastText subword vectors and matched timing

Two follow-up experiments.

**A. Timing on identical tweets.** The earlier efficiency run timed our model
and the transformers on different tweet samples. Here every model is timed on
the same tweets: throughput on all 2,500 test tweets on GPU and on the first
250 on CPU, batch-1 latency on test tweets 3 to 102 (tweets 0 to 2 are
warm-up).

**B. fastText subword vectors instead of random vectors.** The word-only
baseline gives words missing from the fastText `.vec` file a random vector, and
maps test words unseen in training to UNK. The full fastText model (`.bin`)
builds a vector for any word from its character n-grams. Three configurations,
same protocol as the extra baselines (train on the 6,000-tweet training part,
early stopping on the 1,500 validation tweets, up to 60 epochs, patience 12,
batch 32, CRF, five seeds, then score test):

| configuration | question |
|---|---|
| word only, random rows replaced by fastText | does filling the random rows help? |
| word only, fastText for every word (open vocabulary) | does fastText's own subword information match our subword channel? |
| word (open vocabulary) + BPE-1000 pieces | does the subword channel still add to it? (optional, slowest) |

Their controls come from earlier runs with the same protocol and are quoted in
the results cell.

## How to run

1. Settings: Accelerator = GPU T4 x2, Internet = On
2. Leave `SMOKE = True` and Run All (about 20 minutes, most of it downloading
   the 3.2 GB fastText model). Every step runs on a tiny scale and the last
   cell projects the full runtime.
3. Set `SMOKE = False`, then Save Version -> Save & Run All (Commit). Collect
   `/kaggle/working/followup/` when it finishes.

## Settings

In [1]:
SMOKE = False      # True = quick check of every step plus a runtime projection
PART  = 'all'     # 'all', 'timing' or 'fasttext'
INCLUDE_WORD_SUBWORD = True   # word (open vocabulary) + BPE pieces, the slowest configuration

BRANCH = 'sabith'
REPO   = 'https://github.com/hatheem-r/project_DNN.git'

DO_TIMING   = PART in ('all', 'timing')
DO_FASTTEXT = PART in ('all', 'fasttext')
print('SMOKE TEST' if SMOKE else 'FULL RUN', '| timing', DO_TIMING, '| fastText', DO_FASTTEXT)

FULL RUN | timing True | fastText True


## GPU

In [2]:
import torch, os
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
print('CPU cores available:', os.cpu_count())
assert torch.cuda.is_available(), 'Settings -> Accelerator -> GPU T4 x2'

torch 2.10.0+cu128 | CUDA True
Tesla T4
CPU cores available: 4


## Code

In [3]:
import os, shutil, subprocess
os.chdir('/kaggle/working')
if os.path.exists('project'):
    shutil.rmtree('project')
!git clone -q -b $BRANCH $REPO project
os.chdir('/kaggle/working/project')
!pip install -q 'datasets<3.0.0' pytorch-crf sentencepiece 2>&1 | tail -1
os.makedirs('results', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)

def helptext(script):
    return subprocess.run(['python', script, '--help'], capture_output=True, text=True).stdout
assert '--open-vocab' in helptext('notebooks/07_subword_model.py'), 'clone is stale'
assert '--limit' in helptext('notebooks/10_analysis.py'), 'clone is stale'
print('HEAD:', subprocess.run(['git', 'log', '-1', '--oneline'],
                              capture_output=True, text=True).stdout.strip())

gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
HEAD: 1536072 Skip missing files when clearing old timing results


## Tests

The alignment tests check that piece lists line up with words. A break there
shifts labels against tokens without raising an error.

In [4]:
!python tests/test_metrics.py | tail -2
!python tests/test_subword_alignment.py | tail -2


all 9 tests passed
All alignment tests passed.



## Helper

Streams each command's output live and saves it to a log file. `gpu=False`
hides the GPU from that one command, so it runs on the CPU.

In [5]:
import os, subprocess, shutil

OUT = '/kaggle/working/followup'
os.makedirs(OUT, exist_ok=True)

def run(args, logfile, gpu=True):
    env = dict(os.environ)
    if not gpu:
        env['CUDA_VISIBLE_DEVICES'] = ''
    print('$', ' '.join(args), '' if gpu else '   [GPU hidden]', flush=True)
    with open(logfile, 'w') as log:
        p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env=env)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        rc = p.wait()
    print('exit code', rc)
    assert rc == 0, f'failed - see {logfile}'
    if not SMOKE:
        shutil.copy(logfile, OUT)

# A. Timing on identical tweets

Our model is the subword-only configuration, trained for one seed as in the
earlier efficiency run (the weights do not affect timing). Earlier checkpoints
and timing files are deleted first so nothing is reused by accident.

In [6]:
if SMOKE:
    EPOCHS, TRF = 1, ['--model', 'sinbert']
    GPU_LIMIT, CPU_LIMIT, LAT = 64, 16, 3
else:
    EPOCHS, TRF = 35, ['--all']
    GPU_LIMIT, CPU_LIMIT, LAT = 0, 250, 100     # GPU_LIMIT 0 = all 2,500 tweets

if DO_TIMING:
    import glob
    for f in (['artifacts/analysis_model_minimal.pt']
              + glob.glob('results/efficiency_minimal*') + glob.glob('results/transformer_efficiency*')):
        if os.path.exists(f):
            os.remove(f)
            print('removed', f)

    gpu_lim = ['--limit', str(GPU_LIMIT)] if GPU_LIMIT else []
    cpu_lim = ['--limit', str(CPU_LIMIT)]
    lat = ['--latency-tweets', str(LAT)]

    run(['python', 'notebooks/10_analysis.py', '--efficiency', '--minimal',
         '--epochs', str(EPOCHS), *gpu_lim, *lat], 'results/efficiency_minimal_gpu.txt')
    log = open('results/efficiency_minimal_gpu.txt').read()
    assert f'training one seed for {EPOCHS} epochs' in log, 'did not train a fresh model'
    assert 'word-embedding table   none' in log, 'model still has an embedding table'
else:
    print('skipped (PART)')

removed results/efficiency_minimal_cpu_1thread.txt
removed results/efficiency_minimal_gpu.txt
removed results/efficiency_minimal_cpu.txt
removed results/transformer_efficiency_cpu.txt
removed results/transformer_efficiency_gpu.txt
$ python notebooks/10_analysis.py --efficiency --minimal --epochs 35 --latency-tweets 100 

0. SETUP




Generating train split: 100%|██████████| 7500/7500 [00:00<00:00, 34628.53 examples/s]

Generating test split: 100%|██████████| 2500/2500 [00:00<00:00, 44139.60 examples/s]
device cuda   train 7,500   test 2,500
checkpoint: artifacts/analysis_model_minimal.pt
training the tokenizer on the full train split
fastText vectors NOT loaded - the subword-only model has no word channel
model: subword only
tokenizer 1,000 pieces   vocab 33,006

EFFICIENCY MEASUREMENTS - FOR SECTION 9
no checkpoint; training one seed for 35 epochs (about 14 minutes on a T4)
  epoch 10/35  loss 1.5406
  epoch 20/35  loss 1.3384
  epoch 30/35  loss 1.1999
  epoch 35/35  loss 1.1581
save

Our model on CPU, all cores, then one thread. Both reuse the checkpoint above.

In [7]:
if DO_TIMING:
    run(['python', 'notebooks/10_analysis.py', '--efficiency', '--minimal', *cpu_lim, *lat],
        'results/efficiency_minimal_cpu.txt', gpu=False)
    run(['python', 'notebooks/10_analysis.py', '--efficiency', '--minimal', '--threads', '1',
         *cpu_lim, *lat], 'results/efficiency_minimal_cpu_1thread.txt', gpu=False)

$ python notebooks/10_analysis.py --efficiency --minimal --limit 250 --latency-tweets 100    [GPU hidden]

0. SETUP
device cpu   train 7,500   test 2,500
checkpoint: artifacts/analysis_model_minimal.pt
fastText vectors NOT loaded - the subword-only model has no word channel
model: subword only
tokenizer 1,000 pieces   vocab 33,006

EFFICIENCY MEASUREMENTS - FOR SECTION 9
loaded artifacts/analysis_model_minimal.pt
parameters (measured):
  total                             176,058
  trainable                         176,058
  frozen                                  0
  word_embedding                          0
  subword_channel                    90,800
  sentence_head                           0
  lstm_input_dim                        100
  non_embedding_trainable           176,058

inference over test[0:250], measured on cpu:
  tweets                 250
  real tokens            6,079
  wall clock             0.24s
  tweets per second      1,063
  tokens per second      25,847
  ms per

The transformers, with their public weights and an untrained classification
head. Inference only; most of the time is downloading about 5 GB.

In [8]:
if DO_TIMING:
    run(['python', 'notebooks/11_transformer_efficiency.py', *TRF, *gpu_lim, *lat],
        'results/transformer_efficiency_gpu.txt')
    run(['python', 'notebooks/11_transformer_efficiency.py', *TRF, '--allow-cpu', *cpu_lim, *lat],
        'results/transformer_efficiency_cpu.txt', gpu=False)
    for f in ('results/transformer_efficiency_gpu.txt', 'results/transformer_efficiency_cpu.txt'):
        assert 'FAILED' not in open(f).read(), f'a model failed - see {f}'

$ python notebooks/11_transformer_efficiency.py --all --latency-tweets 100 

SETUP
device: cuda   CPU threads used: 2 (of 4 available)
test tweets timed: 2,500

sinbert  (NLPC-UOM/SinBERT-large)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2217.70it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]
RobertaForTokenClassification LOAD REPORT from: NLPC-UOM/SinBERT-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.bias                 | MISSING    | 
classifier.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arc

One table. The check at the top confirms that every model in a column was
timed on the same tweets.

In [9]:
import glob, json

def load(pattern):
    hits = sorted(glob.glob(pattern))
    assert hits, f'no file matches {pattern}'
    return json.load(open(hits[-1]))

if DO_TIMING:
    ours_gpu = load('results/efficiency_minimal_cuda_t*.json')
    ours_cpu = [json.load(open(f)) for f in sorted(glob.glob('results/efficiency_minimal_cpu_t*.json'))]
    ours_cpu_all = max(ours_cpu, key=lambda r: r['threads'])
    ours_cpu_one = min(ours_cpu, key=lambda r: r['threads'])
    trf_gpu = load('results/transformer_efficiency_cuda_t*.json')
    trf_cpu = load('results/transformer_efficiency_cpu_t*.json')

    for col, ours, trf in (('GPU', ours_gpu, trf_gpu), ('CPU', ours_cpu_all, trf_cpu)):
        for key in ('throughput_tweets', 'latency_tweets'):
            sets = {ours[key]} | {r[key] for r in trf}
            assert len(sets) == 1, f'{col} {key} differ: {sets}'
            print(f'{col} {key:<18} every model: {sets.pop()}')
    print()

    cpu_by_name = {r['model']: r for r in trf_cpu}
    rows = [('ours (subword only)', ours_gpu['parameters']['total'], ours_gpu['model_file_mb'],
             '-', ours_gpu, ours_cpu_all)]
    for g in trf_gpu:
        rows.append((g['model'], g['parameters'], g['size_mb'], g['mean_tokens_per_tweet'],
                     g, cpu_by_name.get(g['model'])))

    def tps(r): return f"{r['tweets_per_s']:,.0f}" if r else '-'
    def b1(r):  return f"{r['latency_ms_median']:.2f}" if r and r.get('latency_ms_median') else '-'

    print(f"CPU columns use {ours_cpu_all['threads']} threads for every model.")
    print(f"{'model':<22}{'params':>14}{'MB':>9}{'tok/tw':>8}{'GPU tw/s':>10}{'GPU b1 ms':>11}"
          f"{'CPU tw/s':>10}{'CPU b1 ms':>11}")
    print('-' * 95)
    for name, params, mb, tok, g, c in rows:
        print(f"{name:<22}{params:>14,}{mb:>9.1f}{tok:>8}{tps(g):>10}{b1(g):>11}{tps(c):>10}{b1(c):>11}")
    print(f"\nours on ONE CPU thread: {ours_cpu_one['tweets_per_s']:,.0f} tweets/s, "
          f"batch-1 median {ours_cpu_one['latency_ms_median']:.2f} ms")
    print('tok/tw = mean subword tokens per tweet for each transformer tokenizer')

    if SMOKE:
        print('\nTiming steps ran. These numbers are meaningless (1 epoch, tiny samples).')
    else:
        for f in glob.glob('results/efficiency_minimal*') + glob.glob('results/transformer_efficiency*'):
            shutil.copy(f, OUT)

GPU throughput_tweets  every model: test[0:2500]
GPU latency_tweets     every model: test[3:103]
CPU throughput_tweets  every model: test[0:250]
CPU latency_tweets     every model: test[3:103]

CPU columns use 2 threads for every model.
model                         params       MB  tok/tw  GPU tw/s  GPU b1 ms  CPU tw/s  CPU b1 ms
-----------------------------------------------------------------------------------------------
ours (subword only)          176,058      0.7       -     1,261       4.62     1,063       5.18
sinbert                  125,389,058    501.6    76.3       144       8.73         7     127.12
xlmt                     277,454,594   1109.8    45.6       164       8.72         8      92.89
xlmr-base                277,454,594   1109.8    45.6       166       8.43         9      92.33
xlmr-large               558,842,882   2235.4    45.6        53      16.86         2     364.40

ours on ONE CPU thread: 908 tweets/s, batch-1 median 4.34 ms
tok/tw = mean subword tokens 

# B. fastText subword vectors

The `.vec` file (about 460 MB) builds the baseline matrix, exactly as in the
earlier runs. The `.bin` model (3.2 GB compressed) is decompressed while it
downloads, to `/kaggle/temp`, so neither is saved as output.

In [10]:
if DO_FASTTEXT:
    EMB = '/kaggle/temp/embeddings'
    os.makedirs(EMB, exist_ok=True)
    free_gb = shutil.disk_usage(EMB).free / 1e9
    print(f'free disk at {EMB}: {free_gb:.0f} GB')
    assert free_gb > 12, 'not enough disk for the fastText model'

    VEC = f'{EMB}/cc.si.300.vec.gz'
    if not os.path.exists(VEC):
        !wget -q -O $VEC https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.vec.gz
    os.environ['SOLD_VECTORS'] = VEC
    !python notebooks/03_embeddings.py 2>&1 | tail -4
    assert os.path.exists('artifacts/embedding_matrix.npy'), 'embedding matrix was not built'

    BIN = f'{EMB}/cc.si.300.bin'
    if not os.path.exists(BIN):
        !wget -q -O - https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.bin.gz | gunzip -c > $BIN
    print(f'{BIN}: {os.path.getsize(BIN) / 1e9:.2f} GB')
else:
    print('skipped (PART)')

free disk at /kaggle/temp/embeddings: 1142 GB
test unseen tokens    14.3% never appeared in training
test unseen types     48.7% of distinct words are new
offensive word cov    96.2%

/kaggle/temp/embeddings/cc.si.300.bin: 4.39 GB


The `fasttext` package, needed only to read the `.bin` model.

In [11]:
def has_fasttext():
    return subprocess.run(['python', '-c', 'import fasttext'], capture_output=True).returncode == 0

if DO_FASTTEXT:
    for pkg in ('fasttext', 'fasttext-wheel'):
        if has_fasttext():
            break
        !pip install -q $pkg 2>&1 | tail -2
    assert has_fasttext(), 'could not install fasttext'
    print('fasttext available')

fasttext available


Builds the two matrices and reports, for test word occurrences, how many had a
real vector, a random one, or none (UNK). The `.bin` file is deleted afterwards.

In [12]:
if DO_FASTTEXT:
    run(['python', 'notebooks/12_fasttext_oov.py', '--bin', BIN], 'results/fasttext_oov_prep.txt')
    os.remove(BIN)
    for f in ('artifacts/embedding_matrix_ftfill.npy', 'artifacts/embedding_matrix_ftopen.npy'):
        assert os.path.exists(f), f'{f} was not written'

$ python notebooks/12_fasttext_oov.py --bin /kaggle/temp/embeddings/cc.si.300.bin 

1. LOAD
training-part vocabulary 28,456   matrix (28456, 300)
fastText model: 818,830 words, dimension 300

2. WHICH ROWS ARE RANDOM
real .vec vector   23,314 (81.9%)
random vector      5,140 (18.1%)
  of the random rows, words in fastText's own vocabulary: 4
  the rest get a vector built only from character n-grams

3. MATRICES
artifacts/embedding_matrix_ftfill.npy   (28456, 300)   (5,140 rows replaced)
artifacts/embedding_matrix_ftopen.npy   (40171, 300)   (11,715 validation/test words added)

4. TEST WORD OCCURRENCES BY SOURCE OF THEIR VECTOR
test word occurrences: 59,670
  real .vec vector                  81.4%   all three matrices
  random in baseline                 4.3%   fastText in ftfill and ftopen
  unseen in training (UNK)          14.3%   fastText in ftopen only

5. SANITY CHECK - NEIGHBOURS OF REPLACED WORDS
Nearest real-vector words to the most frequent replaced words. Related
forms here

BPE-1000 tokenizer for the word + pieces configuration, trained on the
6,000-tweet training part with the call `notebooks/06_subword_tokenizer.py`
uses.

In [13]:
if DO_FASTTEXT and INCLUDE_WORD_SUBWORD:
    import sys
    sys.path.insert(0, 'src')
    from data import load_sold, train_val_split
    from subword import train_sentencepiece
    train_part, _ = train_val_split(load_sold('train'))
    if not os.path.exists('artifacts/sp_bpe_1000.model'):
        train_sentencepiece([' '.join(t) for t in train_part['token_list']], 1000,
                            'artifacts/sp_bpe_1000', model_type='bpe')
    print('tokenizer ready:', os.path.exists('artifacts/sp_bpe_1000.model'))

tokenizer ready: True


## Runs

The log and the results CSV are copied to the output folder after every
configuration, so a run that stops early keeps what finished.

In [14]:
FILL = 'artifacts/embedding_matrix_ftfill.npy'
OPEN = 'artifacts/embedding_matrix_ftopen.npy'
CONFIGS = [
    # name,                    --sp,       extra flags
    ('bilstm_word_ftfill',     'none',     ['--matrix', FILL]),
    ('bilstm_word_ftopen',     'none',     ['--matrix', OPEN, '--open-vocab']),
    ('bilstm_word_bpe_ftopen', 'bpe_1000', ['--matrix', OPEN, '--open-vocab']),
]
if not INCLUDE_WORD_SUBWORD:
    CONFIGS = CONFIGS[:2]

CSV = 'results/results_fasttext_smoke.csv' if SMOKE else 'results/results_fasttext_oov.csv'
if os.path.exists(CSV):
    os.remove(CSV)

if DO_FASTTEXT:
    for name, sp, flags in CONFIGS:
        tag = ('smoke_' if SMOKE else '') + name
        args = ['python', 'notebooks/07_subword_model.py', '--final', '--sp', sp,
                *flags, '--tag', tag, '--csv', CSV]
        if SMOKE:
            args += ['--seeds', '1', '--epochs', '2', '--patience', '2']
        run(args, f'results/extra_{tag}.txt')
        if not SMOKE:
            shutil.copy(CSV, OUT)
        print()

$ python notebooks/07_subword_model.py --final --sp none --matrix artifacts/embedding_matrix_ftfill.npy --tag bilstm_word_ftfill --csv results/results_fasttext_oov.csv 

0. SETUP
device cuda   mode FINAL (scores test)
seeds [1, 2, 3, 4, 5]   batch 32   crf True   epochs<=60 patience 12
embedding matrix: artifacts/embedding_matrix_ftfill.npy

row-level results append to: results/results_fasttext_oov.csv
this readable report goes to stdout - redirect it to results/

train-part 6,000  val 1,500  test 2,500
vocab 28,456   embedding matrix (28456, 300)

1. FINAL RUN - none
Frozen Phase 1 settings (batch 32, CRF on) so the number is directly
comparable to the baseline 0.5965.

--- none  (0 pieces)  encoder bilstm  
    tag: bilstm_word_ftfill
    trainable 187,658  (subword channel 0, lstm input 300)
    seed 1  val F1 0.6184   TEST F1 0.6194   (751s, 42 epochs)
    seed 2  val F1 0.6144   TEST F1 0.6110   (982s, 55 epochs)
    seed 3  val F1 0.6227   TEST F1 0.6295   (788s, 44 epochs)
    s

## Results

The controls are earlier runs with the same protocol. A difference larger than
twice the larger standard deviation is reported as clear, larger than one as
modest, and anything smaller as no measurable difference.

In [15]:
import pandas as pd

REF = {   # name: (test F1 mean, sd, source)
    'bilstm_word_only': (0.6012, 0.0050, 'results/extra_bilstm_word_only.txt'),
    'bilstm_bpe_only':  (0.7061, 0.0079, 'results/extra_bilstm_bpe_only.txt'),
    'bilstm_word_bpe':  (0.7104, 0.0069, 'results/step7b_bpe1000.txt, earlier hardware'),
}

if DO_FASTTEXT:
    df = pd.read_csv(CSV)
    df['name'] = df['tag'].str.replace('smoke_', '', regex=False)

    if SMOKE:
        print('SMOKE TEST PASSED - every step ran. Scores are meaningless.\n')
        print('Projected full runtime of part B (5 seeds, about 40 epochs each):')
        total = 0
        for name, g in df.groupby('name', sort=False):
            per_epoch = (g['seconds'] / g['epochs_run']).mean()
            est = per_epoch * 40 * 5 / 60
            total += est
            print(f'  {name:<24} {per_epoch:6.1f} s/epoch  ->  ~{est:4.0f} min')
        print(f'  {"TOTAL":<24} {"":>13}      ~{total:4.0f} min  (~{total / 60:.1f} h)')
        print('\nPart A adds about 45 minutes. Kaggle stops a run at 12 hours; if the')
        print("total is close to that, run PART = 'timing' and PART = 'fasttext' as two commits.")
        print('\nNow set SMOKE = False and use Save Version -> Save & Run All.')
    else:
        agg = df.groupby('name').agg(
            seeds=('seed', 'count'),
            val=('val_f1', 'mean'), val_sd=('val_f1', 'std'),
            test=('test_f1', 'mean'), test_sd=('test_f1', 'std'),
            P=('test_p', 'mean'), R=('test_r', 'mean'))
        print(agg.round(4).to_string(), '\n')
        print('controls:')
        for k, (m, s, src) in REF.items():
            print(f'  {k:<18} {m:.4f} +/- {s:.4f}   ({src})')
        print()

        def get(name):
            if name in agg.index:
                return agg.loc[name, 'test'], agg.loc[name, 'test_sd']
            return REF[name][:2] if name in REF else (None, None)

        def compare(question, a, b):
            (ma, sa), (mb, sb) = get(a), get(b)
            if ma is None or mb is None:
                return
            d = ma - mb
            noise = max(sa, sb, 1e-4)
            if d > 2 * noise:    verdict = 'CLEARLY HIGHER'
            elif d > noise:      verdict = 'modestly higher'
            elif d > -noise:     verdict = 'NO MEASURABLE DIFFERENCE'
            elif d > -2 * noise: verdict = 'modestly lower'
            else:                verdict = 'CLEARLY LOWER'
            print(question)
            print(f'  {a} {ma:.4f} vs {b} {mb:.4f}: {d:+.4f} (larger std {noise:.4f}) -> {verdict}\n')

        compare('Does replacing the random rows with fastText vectors help?',
                'bilstm_word_ftfill', 'bilstm_word_only')
        compare('Does fastText on every word (no UNK) help?',
                'bilstm_word_ftopen', 'bilstm_word_only')
        compare("Our subword channel vs fastText's own subword information:",
                'bilstm_bpe_only', 'bilstm_word_ftopen')
        compare('Does the word channel with fastText on every word add to the pieces?',
                'bilstm_word_bpe_ftopen', 'bilstm_bpe_only')
        compare('Open-vocabulary word channel vs the original word channel, with pieces:',
                'bilstm_word_bpe_ftopen', 'bilstm_word_bpe')
        shutil.copy(CSV, OUT)

                        seeds     val  val_sd    test  test_sd       P       R
name                                                                          
bilstm_word_bpe_ftopen      5  0.7098  0.0037  0.7102   0.0052  0.7482  0.6761
bilstm_word_ftfill          5  0.6174  0.0036  0.6189   0.0070  0.7104  0.5485
bilstm_word_ftopen          5  0.6295  0.0059  0.6148   0.0076  0.7108  0.5418 

controls:
  bilstm_word_only   0.6012 +/- 0.0050   (results/extra_bilstm_word_only.txt)
  bilstm_bpe_only    0.7061 +/- 0.0079   (results/extra_bilstm_bpe_only.txt)
  bilstm_word_bpe    0.7104 +/- 0.0069   (results/step7b_bpe1000.txt, earlier hardware)

Does replacing the random rows with fastText vectors help?
  bilstm_word_ftfill 0.6189 vs bilstm_word_only 0.6012: +0.0177 (larger std 0.0070) -> CLEARLY HIGHER

Does fastText on every word (no UNK) help?
  bilstm_word_ftopen 0.6148 vs bilstm_word_only 0.6012: +0.0136 (larger std 0.0076) -> modestly higher

Our subword channel vs fastText's own su

## Output

In [16]:
if not SMOKE:
    print('saved to', OUT)
    for f in sorted(os.listdir(OUT)):
        print('  ', f)

saved to /kaggle/working/followup
   efficiency_minimal_cpu.txt
   efficiency_minimal_cpu_1thread.txt
   efficiency_minimal_cpu_t1_n250.json
   efficiency_minimal_cpu_t2_n250.json
   efficiency_minimal_cuda_t2.json
   efficiency_minimal_gpu.txt
   extra_bilstm_word_bpe_ftopen.txt
   extra_bilstm_word_ftfill.txt
   extra_bilstm_word_ftopen.txt
   fasttext_oov_prep.txt
   results_fasttext_oov.csv
   transformer_efficiency_cpu.txt
   transformer_efficiency_cpu_t2_n250.json
   transformer_efficiency_cuda_t2.json
   transformer_efficiency_gpu.txt


## What to commit

Everything in `followup/` goes under `results/` in the repo:

```
git add results/efficiency_minimal* results/transformer_efficiency* results/fasttext_oov_prep.txt results/extra_bilstm_word_*ft*.txt results/results_fasttext_oov.csv
git commit -m "Matched timing and fastText subword-vector baselines"
git push
```